# Notebook 02 - Tiền xử lý văn bản

Notebook này tạo dữ liệu sạch cho các bước TF-IDF, K-Means và SVM. Kết quả chính là `datasets/cleaned/train_clean.csv` và `datasets/cleaned/test_clean.csv`.

## 0. Khởi tạo môi trường

In [ ]:
from pathlib import Path
import os
import sys
import warnings

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIG_DIR = Path("results/figures")
CSV_DIR = Path("results/csv")
MODEL_DIR = Path("results/models")
for path in [FIG_DIR, CSV_DIR, MODEL_DIR, Path("datasets/cleaned")]:
    path.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore")

## 2.1 Áp dụng pipeline tiền xử lý

Pipeline chạy đúng thứ tự: lowercase, remove_punctuation, normalize_numbers, remove_stopword, ViTokenizer, remove_repeated_words. Ví dụ dưới đây minh họa rõ tác dụng của từng bước.

In [ ]:
import pandas as pd
from IPython.display import display

from src.preprocessing import (
    add_aspect_columns,
    add_sentiment_columns,
    preprocess_dataframe,
    preprocessing_steps_example,
    read_filestopwords,
)

train_raw = pd.read_csv("datasets/raw/Train.csv", encoding="utf-8")
test_raw = pd.read_csv("datasets/raw/Test.csv", encoding="utf-8")
stop_words = read_filestopwords("datasets/stopwords/vietnamese-stopwords.txt")

example = "Mua máy này 2 tháng pin tụt quá nhanh!!"
display(preprocessing_steps_example(example, stop_words))

## 2.2 Gán nhãn tổng thể

Từ chuỗi aspect-based trong cột `label`, ta đếm `positive_count`, `negative_count`, `neutral_count` rồi gán nhãn tổng thể `sentiment`.

In [ ]:
train_labeled = add_sentiment_columns(train_raw)
test_labeled = add_sentiment_columns(test_raw)

print("Phân phối sentiment sau khi gán nhãn tổng thể - Train:")
display(train_labeled["sentiment"].value_counts().to_frame("count"))
display((train_labeled["sentiment"].value_counts(normalize=True) * 100).round(2).to_frame("percentage"))
print("Nhận xét: Phân phối này quyết định cách đánh giá mô hình; nếu lệch lớp thì F1 Macro quan trọng hơn Accuracy.")

## 2.3 Trích xuất aspect chính

`main_aspect` lấy aspect đầu tiên có ý nghĩa khác `OTHERS`. `aspects_list` giữ toàn bộ cặp `(aspect, sentiment)` để phục vụ EDA và phân tích sau này.

In [ ]:
train_aspect = add_aspect_columns(train_labeled)
display(train_aspect[["label", "sentiment", "main_aspect", "aspects_list"]].head())

aspect_dist = train_aspect["main_aspect"].value_counts()
display(aspect_dist)
print(f"Nhận xét: main_aspect phổ biến nhất là {aspect_dist.index[0]}, đây có thể là chủ đề lớn nhất khi so sánh với K-Means.")

## 2.4 Kiểm tra sau tiền xử lý

Chạy pipeline cho toàn bộ train/test, kiểm tra số dòng rỗng sau clean, phân phối độ dài và một vài ví dụ trước/sau.

In [ ]:
train_clean_full = preprocess_dataframe(train_raw, stop_words)
test_clean_full = preprocess_dataframe(test_raw, stop_words)

empty_train = (train_clean_full["comment"].str.strip() == "").sum()
empty_test = (test_clean_full["comment"].str.strip() == "").sum()
print(f"Số dòng train rỗng sau clean: {empty_train}")
print(f"Số dòng test rỗng sau clean : {empty_test}")

train_clean_full = train_clean_full[train_clean_full["comment"].str.strip() != ""].copy()
test_clean_full = test_clean_full[test_clean_full["comment"].str.strip() != ""].copy()

display(train_clean_full["word_count"].describe().round(2))
display(train_clean_full[["raw_comment", "comment", "sentiment", "main_aspect"]].head(8))
print("Nhận xét: Sau tiền xử lý, câu ngắn gọn hơn, số được chuẩn hóa thành `number`, và các cụm tiếng Việt có thể được nối bằng dấu gạch dưới.")

## 2.5 Lưu dữ liệu sạch

In [ ]:
keep_cols = ["comment", "sentiment", "main_aspect", "n_star", "aspects_list"]
train_clean_full[keep_cols].to_csv("datasets/cleaned/train_clean.csv", index=False, encoding="utf-8-sig")
test_clean_full[keep_cols].to_csv("datasets/cleaned/test_clean.csv", index=False, encoding="utf-8-sig")

print("Đã lưu datasets/cleaned/train_clean.csv:", train_clean_full[keep_cols].shape)
print("Đã lưu datasets/cleaned/test_clean.csv :", test_clean_full[keep_cols].shape)